# 🛡️ D4S: Robust DCT-Watermarking Grand Challenge

Welcome to the **D4S Grand Challenge**! In this assignment, you will design and implement a robust, stealthy, and blind **Discrete Cosine Transform (DCT)-based image watermarking system** in Python.

### 🎯 Objective
Your goal is to implement `watermark_encoder` and `watermark_decoder` so that:
- **File I/O is PNG-based:** Both functions take **PNG** files as input and the encoding produces **PNG** files as output.
- **Embedding & extraction happen in the DCT domain:** Internally, your watermark embedding and recovery logic must transform into and operate within the **DCT domain** rather than relying solely on spatial-domain modifications.
- **Robustness & imperceptibility are balanced:** Your watermark must survive standard lossy JPEG compression attacks and achieve **Zero Errors (BER = 0)** decoding the watermark, while keeping perceptual distortion (**LPIPS**) as low as possible.

> **⚠️ Important Disclaimer:** All watermark embedding **must be performed in the DCT domain**. Submissions that do not truly embed in the DCT domain, or that otherwise violate the challenge rules or required function interfaces, may be **disqualified during manual post-challenge review**, even if they perform well or rank near the top on the automated leaderboard.

Good luck, and may the best algorithm top the leaderboard! 🏆

## 💡 Before We Start

Run the cells below to prepare your environment. Feel free to inspect any of the code! This setup will:
* **Install and Import**: Load required Python libraries (like `lpips` for perceptual quality).
* **Load Public Test Data**: Download the public test images into the `public_test/` folder so you can test your algorithm locally.
* **Define the Evaluator**: Set up the evaluation pipeline (which mirrors the secret evaluation backend) to measure **LPIPS**, and **Bit Error Rate (BER)**.
* **Prepare the Leaderboard Tool**: Define the function used to fetch and display the class standings (you can run it in the leaderboard section below), and inspect your submission results.

> 💡 **Environment Tip:** You are welcome to develop and test locally, **but we strongly advise running and testing your solution in this Google Colab Notebook** so you can verify compatibility with the automated evaluation environment (dependencies, paths, and runtime behavior).

In [ ]:
# Install LPIPS metric and jpegio
!pip install -q lpips pyjpegio
# Some useful imports
# If you use any others, make sure to include them in your submission
import skimage.data
import matplotlib.pyplot as plt
import math
import tempfile
import os
import io
import jpegio as jio
import glob
import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
import lpips
import hashlib
import hmac
import struct
import pandas as pd
from IPython.display import display, HTML
import gdown
import importlib.util
import sys
import zipfile
import time
import warnings
from datetime import datetime, timezone, timedelta
warnings.filterwarnings('ignore')

print("Environment setup complete!")

In [ ]:
# ==========================================
# AUXILIARY IMAGES
# Loading some auxiliary images to public_test/
# to test your watermarking algorithm
# ==========================================
public_test_zip_url = "https://drive.google.com/file/d/1fNUK48LZm0hvkgnH1_X-B9bYf0FnNmG4/view?usp=sharing"
# Create a public test directory and populate it with 10 sample images
os.makedirs("public_test", exist_ok=True)
output_zip = "public_test.zip"

print("Downloading public test images...")
gdown.download(public_test_zip_url, output_zip, quiet=False, fuzzy=True)

# Extract the zip contents into the public_test directory
if os.path.exists(output_zip):
    with zipfile.ZipFile(output_zip, 'r') as zip_ref:
        zip_ref.extractall("public_test")
    # Clean up the zip file
    os.remove(output_zip)
    print("Extraction complete!")
else:
    print("⚠️ Download failed. Please check the file permissions or ID.")

def display_test_images(image_dir="public_test"):
    """Displays all test images in a subplot grid."""
    image_paths = sorted(glob.glob(os.path.join(image_dir, "*.*")))

    if not image_paths:
        print(f"No images found in '{image_dir}'.")
        return

    cols = 5
    rows = math.ceil(len(image_paths) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 3))
    axes = axes.flatten()

    for i, path in enumerate(image_paths):
        img = Image.open(path)
        axes[i].imshow(img)
        axes[i].set_title(os.path.basename(path).split('.')[0], fontsize=10)
        axes[i].axis('off')

    # Hide any unused subplot axes if total images < rows*cols
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# Count only files ending with .png (case-insensitive)
public_images_count = len([
    f for f in os.listdir('public_test')
    if f.lower().endswith(('.png'))
])

print(f"Successfully loaded {public_images_count} public test images into 'public_test/' folder.")
display_test_images()

In [ ]:
# ==========================================
# DISPLAY FUNCTION
# Visualizes original, watermarked, and the
# absolute difference map side-by-side (display_images=True)
# ==========================================
def plot_watermark_comparison(original_path, watermarked_path, strength, lpips_val):
    """Plots a 3-panel comparison of the original, watermarked, and amplified difference map."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    orig_img = Image.open(original_path).convert("RGB")
    watermarked_img = Image.open(watermarked_path).convert("RGB")

    # 1. Original Cover Image
    axes[0].imshow(orig_img)
    axes[0].set_title("Original Cover Image", fontsize=12)
    axes[0].axis('off')

    # 2. Watermarked Image
    axes[1].imshow(watermarked_img)
    axes[1].set_title(f"Watermarked (Strength={strength:.1f})\nLPIPS: {lpips_val:.4f}", fontsize=11)
    axes[1].axis('off')

    # 3. Difference Map (Amplified for visibility)
    orig_np = np.array(orig_img, dtype=np.float32)
    watermarked_np = np.array(watermarked_img, dtype=np.float32)

    # Calculate absolute difference and amplify it (multiply by 10) so subtle changes pop out
    diff = np.abs(orig_np - watermarked_np)
    diff_amplified = np.clip(diff * 10.0, 0, 255).astype(np.uint8)

    axes[2].imshow(diff_amplified)
    axes[2].set_title("Amplified Difference Map\n(10x Boost)", fontsize=11)
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==========================================
# EVALUATION PIPELINE (DO NOT EDIT)
# This script tests your encoder and decoder against
# fixed image transformations and calculates your scores.
# ==========================================
class WatermarkEvaluator:
    def __init__(self, image_dir="public_test"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device.upper()} for evaluation.")

        # Load LPIPS perceptual loss model
        self.lpips_model = lpips.LPIPS(net='alex').to(self.device)
        self.lpips_model.eval()

        self.image_paths = glob.glob(os.path.join(image_dir, "*.*"))

    def apply_attacks(self, quality_factor: int, image_path: str, attacked_output_path: str) -> None:
        """Applies standard JPEG compression attack in-memory, then saves the degraded result as a PNG."""
        img = Image.open(image_path).convert("RGB")
        # 1. Simulate JPEG compression loss using an in-memory buffer
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality_factor)
        buffer.seek(0)
        # 2. Decompress/reload the JPEG-compressed data back into a PIL Image
        attacked_img = Image.open(buffer).convert("RGB")
        # 3. Save the final attacked output path as a PNG
        attacked_img.save(attacked_output_path, format="PNG")

    def run_evaluation(self, embedded_message=1024*[0], password="secret_password", display_images=False):
        # Define the search bounds for strength (e.g., 0 to 250)
        min_strength = 0
        max_strength = 250

        total_optimal_lpips = 0.0
        successful_images = 0
        had_exceptions = False
        num_images = len(self.image_paths)
        to_tensor = T.ToTensor()

        if num_images == 0:
            print("Error: No test images found!")
            return float("nan"), 0, False

        print(f"\nFinding minimum watermark strength (0 to 250) using Binary Search until Zero Errors (BER=0)...")

        with tempfile.TemporaryDirectory() as tmpdir:
            for img_idx, path in enumerate(self.image_paths, start=1):
                base_name = os.path.basename(path)
                cover_img = Image.open(path).convert('RGB')
                cover_tensor = to_tensor(cover_img).unsqueeze(0).to(self.device)

                print(f"\n[{img_idx}/{num_images}] Image: {base_name}")

                for quality_factor in [90, 70]:
                    best_lpips_for_image = None
                    achieved_zero_ber = False

                    # --- BINARY SEARCH FOR MINIMUM STRENGTH ---
                    low = min_strength
                    high = max_strength
                    found_strength = None

                    while low <= high:
                        mid = (low + high) // 2
                        strength = float(mid)

                        try:
                            watermarked_path = os.path.join(tmpdir, f"watermarked_{strength:.2f}_{base_name}")
                            attacked_path = os.path.join(tmpdir, f"attacked_{strength:.2f}_{base_name}")

                            # 1. Student encodes message
                            t0 = time.time()
                            watermark_encoder(path, watermarked_path, embedded_message, strength=strength, password=password)
                            t_enc = time.time() - t0

                            # 2. Apply JPEG compression attack (in-memory -> PNG)
                            self.apply_attacks(quality_factor, watermarked_path, attacked_path)

                            # 3. Student decodes message
                            t0 = time.time()
                            extracted_bits = watermark_decoder(attacked_path, message_length=len(embedded_message), strength=strength, password=password)
                            t_dec = time.time() - t0

                            # 4. Compute Bit Error Rate (BER)
                            orig = np.array(embedded_message)
                            ext = np.array(extracted_bits)
                            min_len = min(len(orig), len(ext))
                            n_err = int(np.sum(orig[:min_len] != ext[:min_len])) if min_len > 0 else len(orig)
                            ber = n_err / min_len if min_len > 0 else 1.0

                            print(f"   [QF={quality_factor}] strength={strength:5.1f} | bit errors={n_err:4d}/{len(orig)} "
                                  f"| BER={ber:.4f} | enc={t_enc:.2f}s dec={t_dec:.2f}s", end="")

                            if ber == 0.0:
                                print(" -> SUCCESS (Trying lower strength)")
                                found_strength = strength
                                high = mid - 1  # Try to find an even lower working strength
                            else:
                                print(" -> too weak (Trying higher strength)")
                                low = mid + 1   # Need more strength

                        except Exception as e:
                            had_exceptions = True
                            print(f"   [QF={quality_factor}] strength={strength:5.1f} | EXCEPTION: {type(e).__name__}: {e}")
                            # If an exception occurs, treat it as failure and try higher strength
                            low = mid + 1

                    # After binary search finishes, if we found a working strength, evaluate its final LPIPS
                    if found_strength is not None:
                        try:
                            strength = found_strength
                            achieved_zero_ber = True

                            # Re-generate the winning watermarked image to measure LPIPS
                            watermarked_path = os.path.join(tmpdir, f"watermarked_{strength:.2f}_{base_name}")
                            watermark_encoder(path, watermarked_path, embedded_message, strength=strength, password=password)

                            watermarked_img_loaded = Image.open(watermarked_path).convert('RGB')
                            watermarked_tensor = to_tensor(watermarked_img_loaded).unsqueeze(0).to(self.device)

                            lpips_val = self.lpips_model(
                                (cover_tensor * 2.0 - 1.0),
                                (watermarked_tensor * 2.0 - 1.0)
                            ).item()

                            best_lpips_for_image = lpips_val
                            print(f" -> {base_name} [QF={quality_factor}]: Min Zero Errors Strength = {strength:.1f} | LPIPS = {lpips_val:.4f}")

                            if display_images:
                                plot_watermark_comparison(path, watermarked_path, strength, lpips_val)
                        except Exception as e:
                            had_exceptions = True
                            achieved_zero_ber = False
                            print(f" -> {base_name} [QF={quality_factor}]: EXCEPTION during LPIPS/finalization: {type(e).__name__}: {e}")
                    else:
                        print(f" -> {base_name} [QF={quality_factor}]: Failed to reach Zero Errors within range [0, 250].")

                    if achieved_zero_ber:
                        total_optimal_lpips += best_lpips_for_image
                        successful_images += 1

        avg_lpips = total_optimal_lpips / successful_images if successful_images > 0 else float("nan")

        print("\n" + "=" * 30)
        print("      EVALUATION RESULTS      ")
        print("=" * 30)
        print(f"Evaluated Images       : {num_images}")
        print(f"Successful Zero Errors : {successful_images}/{num_images*2}")
        print(f"Mean Optimal LPIPS     : {avg_lpips:.4f}  (Only in Zero Errors, lower is better)")
        print(f"Exceptions Occurred    : {had_exceptions}")
        print("=" * 30)
        return avg_lpips, successful_images, had_exceptions

In [ ]:
# ==========================================
# LEADERBOARD VISUALIZER
# This script downloads the current leaderboard
# and displays the Zero-BER rankings and error list in UTC+1
# ==========================================
def view_leaderboard():
    """
    Fetches and displays the live class leaderboard, keeping only the best
    submission per pseudonym, converting all displayed times from UTC to UTC+1.
    """
    url = "https://drive.google.com/file/d/1dV20u0R8MfWDzJ62r_LjRDfylyK7lXRs/view?usp=sharing"
    output_path = "leaderboard.csv"
    UTC_PLUS_1 = timezone(timedelta(hours=1))

    try:
        # Download the latest CSV from Google Drive
        gdown.download(url, output_path, quiet=True, fuzzy=True)

        # Get the file's last modified time and convert from UTC to UTC+1
        if os.path.exists(output_path):
            mod_time = os.path.getmtime(output_path)
            last_updated = datetime.fromtimestamp(mod_time, tz=timezone.utc).astimezone(UTC_PLUS_1).strftime('%Y-%m-%d %H:%M:%S UTC+1')
        else:
            last_updated = "Unknown"

        df = pd.read_csv(output_path)

        # Normalize legacy "Student" column to "Pseudonym" if needed
        if "Pseudonym" not in df.columns and "Student" in df.columns:
            df = df.rename(columns={"Student": "Pseudonym"})

        # Drop Filename and Issues / Notes so the public leaderboard stays clean
        cols_to_hide = [c for c in ["Filename", "Issues / Notes"] if c in df.columns]
        if cols_to_hide:
            df = df.drop(columns=cols_to_hide)

        display(HTML("<h3>🏆 Live Class Watermark Challenge Leaderboard 🏆</h3>"))
        display(HTML(f"<p style='color: #9ca3af; font-family: sans-serif; font-size: 13px;'>🕒 Last Updated: <b>{last_updated}</b> (Grading runs daily)</p>"))

        if df.empty or "Pseudonym" not in df.columns:
            display(HTML("<p style='font-family: sans-serif; color: #6b7280;'><i>No submissions found on the leaderboard yet.</i></p>"))
            return

        # Clean whitespace on Pseudonym and add a normalized key for deduplication
        df["Pseudonym"] = df["Pseudonym"].astype(str).str.strip()
        df["_pseudonym_key"] = df["Pseudonym"].str.lower()

        # Ensure correct data types for sorting columns
        if "Zero Errors" in df.columns:
            df["Zero Errors"] = df["Zero Errors"].astype(str).str.lower() == "true"
        else:
            df["Zero Errors"] = False

        # Parse UTC Submission Time and convert to UTC+1 for display
        if "Submission Time" in df.columns:
            df["Submission Time"] = pd.to_datetime(df["Submission Time"], errors="coerce")
            df["Submission Time"] = df["Submission Time"].dt.tz_localize('UTC').dt.tz_convert(UTC_PLUS_1)
        else:
            df["Submission Time"] = pd.NaT

        # Create a temporary numeric sorting column for LPIPS (using 999.0 only under the hood)
        if "Mean LPIPS" in df.columns:
            df["LPIPS_sort"] = pd.to_numeric(df["Mean LPIPS"], errors="coerce").fillna(999.0)
        else:
            df["Mean LPIPS"] = "-"
            df["LPIPS_sort"] = 999.0

        # --- FILTER BEST SUBMISSION PER PSEUDONYM ---
        # Sort criteria: Zero Errors (desc), LPIPS_sort (asc), Submission Time (desc)
        df_sorted = df.sort_values(
            by=["Zero Errors", "LPIPS_sort", "Submission Time"],
            ascending=[False, True, False]
        )

        # Keep only the top entry per pseudonym (case-insensitive)
        df_best = df_sorted.drop_duplicates(subset=["_pseudonym_key"], keep="first").copy()

        # Clean up temporary helper columns
        df_best = df_best.drop(columns=["LPIPS_sort", "_pseudonym_key"], errors="ignore")

        # Convert Submission Time back to string format in UTC+1 for clean display
        if "Submission Time" in df_best.columns:
            df_best["Submission Time"] = df_best["Submission Time"].dt.strftime('%Y-%m-%d %H:%M:%S')

        # Split into Passed (Zero Errors == True) and Failed/Errors
        df_passed = df_best[df_best["Zero Errors"] == True].drop(columns=["Zero Errors"]).copy()
        df_failed = df_best[df_best["Zero Errors"] == False].drop(columns=["Zero Errors"]).copy()

        # Format Mean LPIPS cleanly in both tables
        if not df_passed.empty and "Mean LPIPS" in df_passed.columns:
            df_passed["Mean LPIPS"] = pd.to_numeric(df_passed["Mean LPIPS"], errors="coerce").map(
                lambda x: f"{x:.4f}" if pd.notna(x) else "-"
            )

        if not df_failed.empty and "Mean LPIPS" in df_failed.columns:
            df_failed["Mean LPIPS"] = pd.to_numeric(df_failed["Mean LPIPS"], errors="coerce").map(
                lambda x: "-" if pd.isna(x) or x >= 999.0 else f"{x:.4f}"
            )
            df_failed = df_failed.rename(columns={"Mean LPIPS": "Mean LPIPS*"})

        # Helper to move timestamp column to the very end of a DataFrame
        def move_timestamp_to_end(dataframe, time_col_name="Submission Time"):
            if time_col_name in dataframe.columns:
                cols = [c for c in dataframe.columns if c != time_col_name] + [time_col_name]
                return dataframe[cols]
            return dataframe

        # --- 1. RENDER ZERO ERRORS LEADERBOARD ---
        if not df_passed.empty:
            df_passed = df_passed.sort_values(by=["Mean LPIPS", "Submission Time"], ascending=[True, False]).reset_index(drop=True)
            df_passed.index = range(1, len(df_passed) + 1)
            df_passed = df_passed.reset_index().rename(columns={"index": "Rank"})
            df_passed = move_timestamp_to_end(df_passed)

            display(HTML("<h4>✨ Qualified Submissions (Zero Errors)</h4>"))
            styled_passed = df_passed.style.set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', '#ffffff'), ('font-family', 'sans-serif'), ('padding', '10px'), ('text-align', 'center')]},
                {'selector': 'td', 'props': [('font-family', 'sans-serif'), ('color', '#111827'), ('padding', '8px'), ('text-align', 'center')]},
                {'selector': 'tr:nth-child(odd)', 'props': [('background-color', '#ffffff')]},
                {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f3f4f6')]}
            ]).hide(axis="index")
            display(styled_passed)
        else:
            display(HTML("<p style='font-family: sans-serif; color: #6b7280;'><i>No submissions have achieved Zero Errors yet. Keep tweaking your code!</i></p>"))

        # --- 2. RENDER SUBMISSIONS WITH ERRORS ---
        if not df_failed.empty:
            df_failed = move_timestamp_to_end(df_failed)
            display(HTML("<br><h4>⚠️ Submissions With Evaluation Errors / Non-Zero Errors</h4>"))
            styled_failed = df_failed.style.set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#9ca3af'), ('color', '#ffffff'), ('font-family', 'sans-serif'), ('padding', '8px'), ('text-align', 'center')]},
                {'selector': 'td', 'props': [('font-family', 'sans-serif'), ('color', '#4b5563'), ('padding', '6px'), ('text-align', 'center')]},
                {'selector': 'tr:nth-child(odd)', 'props': [('background-color', '#f9fafb')]},
                {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f3f4f6')]}
            ]).hide(axis="index")
            display(styled_failed)
            display(HTML("<p style='color: #6b7280; font-family: sans-serif; font-size: 11px; margin-top: 4px;'><i>*Mean LPIPS is calculated only on successfully processed images. All times shown in UTC+1.</i></p>"))

    except Exception as e:
        print(f"⚠️ Could not load leaderboard at this moment: {e}")

In [ ]:
# ==========================================
# PARTICIPANT SUBMISSION HISTORY VIEWER
# ==========================================
def see_all_submissions(pseudonym: str):
    """
    Fetches the leaderboard and displays all historical submissions for a specific pseudonym,
    ordered chronologically (Attempt 1 is the oldest submission), converting times from UTC to UTC+1
    and showing diagnostic notes for failed or skipped runs.
    """
    url = "https://drive.google.com/file/d/1dV20u0R8MfWDzJ62r_LjRDfylyK7lXRs/view?usp=sharing"
    output_path = "leaderboard.csv"
    UTC_PLUS_1 = timezone(timedelta(hours=1))

    try:
        # Download the latest CSV from Google Drive
        gdown.download(url, output_path, quiet=True, fuzzy=True)

        if os.path.exists(output_path):
            df = pd.read_csv(output_path)
        else:
            print("⚠️ Leaderboard file not found.")
            return

        # Normalize legacy "Student" column to "Pseudonym" if needed
        if "Pseudonym" not in df.columns and "Student" in df.columns:
            df = df.rename(columns={"Student": "Pseudonym"})

        if df.empty or "Pseudonym" not in df.columns:
            display(HTML("<p style='font-family: sans-serif; color: #6b7280;'><i>No submissions found.</i></p>"))
            return

        clean_pseudonym = str(pseudonym).strip()

        # Filter for the specific pseudonym (case-insensitive and whitespace-stripped match)
        participant_df = df[df["Pseudonym"].astype(str).str.strip().str.lower() == clean_pseudonym.lower()].copy()

        if participant_df.empty:
            display(HTML(f"<p style='font-family: sans-serif; color: #6b7280;'><i>No submission history found for: <b>{clean_pseudonym}</b>. Make sure your PSEUDONYM variable matches!</i></p>"))
            return

        # Convert stored UTC Submission Time to UTC+1 for chronological sorting and display
        if "Submission Time" in participant_df.columns:
            participant_df["Submission Time"] = pd.to_datetime(participant_df["Submission Time"], errors="coerce")
            participant_df["Submission Time"] = participant_df["Submission Time"].dt.tz_localize('UTC').dt.tz_convert(UTC_PLUS_1)
            participant_df = participant_df.sort_values(by="Submission Time", ascending=True).reset_index(drop=True)
            participant_df["Submission Time"] = participant_df["Submission Time"].dt.strftime('%Y-%m-%d %H:%M:%S')

        # Drop the Filename column if present in legacy CSVs so the table stays clean
        if "Filename" in participant_df.columns:
            participant_df = participant_df.drop(columns=["Filename"])

        # Format Mean LPIPS for display (handle missing/999 cleanly)
        if "Mean LPIPS" in participant_df.columns:
            participant_df["Mean LPIPS"] = pd.to_numeric(participant_df["Mean LPIPS"], errors="coerce").map(
                lambda x: "-" if pd.isna(x) or x >= 999.0 else f"{x:.4f}"
            )

        # Clean up Issues / Notes for display (replace NaN/empty with "-")
        if "Issues / Notes" in participant_df.columns:
            participant_df["Issues / Notes"] = participant_df["Issues / Notes"].apply(
                lambda x: "-" if pd.isna(x) or str(x).strip() == "" else str(x)
            )

        display(HTML(f"<h3>📜 Submission History for: {clean_pseudonym}</h3>"))
        display(HTML("<p style='color: #9ca3af; font-family: sans-serif; font-size: 13px;'>Showing all historical entries ordered chronologically from oldest to newest.</p>"))

        # Add an attempt number index starting from 1 (1 = first ever attempt)
        participant_df.index = range(1, len(participant_df) + 1)
        participant_df = participant_df.reset_index().rename(columns={"index": "Attempt"})

        styled_participant = participant_df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#1f2937'), ('color', '#ffffff'), ('font-family', 'sans-serif'), ('padding', '10px'), ('text-align', 'center')]},
            {'selector': 'td', 'props': [('font-family', 'sans-serif'), ('color', '#111827'), ('padding', '8px'), ('text-align', 'center')]},
            {'selector': 'tr:nth-child(odd)', 'props': [('background-color', '#ffffff')]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f3f4f6')]}
        ]).hide(axis="index")
        display(styled_participant)

        display(HTML("<p style='color: #6b7280; font-family: sans-serif; font-size: 11px; margin-top: 4px;'><i>*Mean LPIPS is calculated only on successfully processed images. All times shown in UTC+1.</i></p>"))

    except Exception as e:
        print(f"⚠️ Could not load submission history: {e}")

## 🏆 Leaderboard 🏆
Run the cell below to check the live class rankings!

> 💡 **Note**: The automated grading backend runs once daily. If you recently uploaded your submission file through the **Grand Challenge Form**, your new score and **Zero Errors** status will appear on the leaderboard after the next daily update. Additionally, each participant is limited to **at most 2 submissions per daily grading run**, any submissions beyond the first two **will not be evaluated**, and will need to be reuploaded on the next day.

> 💡 **Note**: The leaderboard shows only the best submission for each student. If your newest submission doesn't appear on the leaderboard after an update, it either didn't improve your best score or encountered an error during evaluation. You can inspect all of your submissions with `see_all_submissions("<pseudonym>")`.

> **⚠️ Important Disclaimer**: This leaderboard is generated **automatically** based on the evaluation pipeline and metrics. Ranking high or achieving top scores on the leaderboard **does not guarantee** that your submission complies with all challenge requirements (such as DCT-domain embedding and valid function interfaces). Any submission found not to follow the rules may be **disqualified** during review.

In [ ]:
view_leaderboard()
# see_all_submissions("<pseudonym>")

## 📝 Your Code Implementation

Follow the guidelines below to build your DCT watermarking algorithm:

1. **Set your Pseudonym**: Define a variable `PSEUDONYM="<pseudonym>"` at the very top of your code cell. The automated grading backend uses this to attribute your score on the leaderboard. You must keep your pseudonym the same across all submissions.
2. **Required Functions**: You **must** keep the function headers for `watermark_encoder` and `watermark_decoder` intact.
   > **Note on `strength` in `watermark_decoder`:** It is **not mandatory to use** `strength` in the decoder, but the parameter **must still remain in the `watermark_decoder` function header** so the automated evaluation can call it without signature errors.
3. **Helper Functions**: You are free to write internal helper functions, but make sure all necessary modules (like `numpy`, `cv2`, `jpegio`, etc.) are imported in your code and available in the environment.

In [ ]:
# ==========================================
# STUDENT SUBMISSION
# ==========================================
PSEUDONYM = "<pseudonym>"

def watermark_encoder(
    cover_image_path: str,
    output_image_path: str,
    message_bits: list,
    strength: float,
    password: str
) -> None:

    #### YOUR CODE HERE ####

    return


def watermark_decoder(
    watermarked_image_path: str,
    message_length: int,
    strength: float,
    password: str
) -> list:

    #### YOUR CODE HERE ####

    return extracted_bits


## 🔍 Local Self-Evaluation

Before submitting, run the cell below to test your `watermark_encoder` and `watermark_decoder` against the public test dataset (`public_test/`).

- **How it works**: The evaluator searches the `strength` parameter across a range from **0 to 250** using **binary search** (with step resolution 1) and applies simulated JPEG compression attacks (QF-90, QF-70) to find the **lowest possible strength** that decodes the watermark with **Zero Errors (BER = 0)**.
- **Using `strength` effectively**: You can inspect which `strength` value the evaluator selects during runs and adapt how your encoder uses `strength` in your code to optimize this search and minimize perceptual perturbation on the image.
- **Scoring**: Your final metric on the leaderboard will be the **Mean LPIPS** (lower is better, meaning higher perceptual quality) across all test images if you successfully hit Zero Errors.
- **Customizing Tests**: Feel free to experiment by tweaking the `embedded_message` (keep it to 1024 bits) and `password` parameters in the evaluation call below. *Note: During the actual grading pipeline, these will be replaced with secret, fixed values to ensure blind evaluation.*
- **Visual Inspection**: If you want to visually inspect how your watermarked images compare against the originals, you can change `display_images=False` to `display_images=True` in the evaluation call below.

In [ ]:
# ==========================================
# SELF EVALUATION
# ==========================================
evaluator = WatermarkEvaluator()
evaluator.run_evaluation(embedded_message=[1, 0, 1, 1, 0, 0, 1, 0] * 128, password="secret_password", display_images=False)

## 🚀 How to Submit Your Solution

Once you are happy with your local evaluation results, follow these steps to submit your solution:

1. **Prepare your standalone `.py` file**: Export or save your implementation as a single Python file named `<pseudonym>.py` (for example, if your pseudonym is `Blue Fox`, name the file `blue_fox.py`).
2. **Upload to the Google Form**: Submit your `.py` file through the Google Form for the Grand Challenge.
3. **Check the Leaderboard Tomorrow**: The automated grading backend runs once daily. Scroll back up to the **Leaderboard** section and re-run `view_leaderboard()` after the next update to see your live rank and **Zero Errors** status.

> **⚠️ Before submitting, make sure that:**
> 1. **File name**: You upload a single Python file named `<pseudonym>.py` (e.g., `blue_fox.py` for pseudonym `Blue Fox`). Use **only lowercase letters, digits, and underscores**.
> 2. **Required contents**: The file defines all of the following:
>    - A variable `PSEUDONYM = "<pseudonym>"` matching the pseudonym in your file name.
>    - A function `watermark_encoder(cover_image_path, output_image_path, message_bits, strength, password)`.
>    - A function `watermark_decoder(watermarked_image_path, message_length, strength, password)`.
> 3. **Self-contained**: The file runs on its own, with every required import and helper function included. Make sure all imported modules are available in the shared Colab environment.
> 4. **Tested**: You ran the code with the provided local self-evaluation and it recovered the message with **Zero Errors** (BER = 0) on the images you tested.

### 📊 Resubmissions and Grading
- The automated grading backend runs **once per day**.
- At most **two** uploaded submissions per participant are evaluated between grading runs until the deadline; if more than two are sent between runs, the extra ones will not be evaluated.
- The leaderboard displays your **best result** among the submissions that were evaluated.
- If your newest submission does not appear on the leaderboard after an update, it either did not beat your previous best score or encountered an error during evaluation. Run `see_all_submissions("<pseudonym>")` in the Colab notebook to inspect your full submission history.